# Bench Scripts Merged (auto-generated)

이 노트북은 bench 폴더의 benchmark.py 제외 스크립트를 순서대로 병합한 자동 생성본입니다.

## File: compare_models.py

In [ ]:
#!/usr/bin/env python3
"""Run benchmark.py across multiple GGUF models and compare results.

This script executes each model in a separate process to avoid memory carry-over.
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import subprocess
import sys
import tempfile
import time
from dataclasses import dataclass
from typing import List, Optional


@dataclass
class ModelSpec:
    name: str
    path: str
    n_gpu_layers: Optional[int] = None
    n_ctx: Optional[int] = None
    max_tokens: Optional[int] = None


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Compare multiple GGUF models on same prompts.")
    parser.add_argument(
        "--models-file",
        required=True,
        help="CSV file with columns: name,path[,n_gpu_layers,n_ctx,max_tokens]",
    )
    parser.add_argument("--prompts-file", default=None)
    parser.add_argument("--num-prompts", type=int, default=3)
    parser.add_argument("--max-tokens", type=int, default=128)
    parser.add_argument("--n-ctx", type=int, default=1024)
    parser.add_argument("--n-gpu-layers", type=int, default=35)
    parser.add_argument("--n-threads", type=int, default=min(6, os.cpu_count() or 1))
    parser.add_argument("--temperature", type=float, default=0.7)
    parser.add_argument("--top-p", type=float, default=0.9)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--warmup", action="store_true")
    parser.add_argument("--output-csv", default="outputs/comparison_results.csv")
    parser.add_argument("--fail-fast", action="store_true")
    return parser.parse_args()


def to_optional_int(value: Optional[str]) -> Optional[int]:
    if value is None:
        return None
    value = value.strip()
    if not value:
        return None
    return int(value)


def load_model_specs(models_file: str) -> List[ModelSpec]:
    if not os.path.isfile(models_file):
        raise FileNotFoundError(f"Models CSV not found: {models_file}")

    specs: List[ModelSpec] = []
    with open(models_file, "r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames is None:
            raise ValueError("Models CSV is empty.")

        fieldnames = {name.strip() for name in reader.fieldnames}
        required = {"name", "path"}
        if not required.issubset(fieldnames):
            raise ValueError("Models CSV must include headers: name,path")

        for row in reader:
            name = (row.get("name") or "").strip()
            path = (row.get("path") or "").strip()
            if not name or not path:
                continue
            specs.append(
                ModelSpec(
                    name=name,
                    path=path,
                    n_gpu_layers=to_optional_int(row.get("n_gpu_layers")),
                    n_ctx=to_optional_int(row.get("n_ctx")),
                    max_tokens=to_optional_int(row.get("max_tokens")),
                )
            )

    if not specs:
        raise ValueError(f"No valid model rows found in {models_file}")

    return specs


def build_cmd(
    benchmark_script: str,
    spec: ModelSpec,
    args: argparse.Namespace,
    json_out: str,
    n_gpu_layers_override: Optional[int] = None,
) -> List[str]:
    model_max_tokens = spec.max_tokens if spec.max_tokens is not None else args.max_tokens
    model_n_ctx = spec.n_ctx if spec.n_ctx is not None else args.n_ctx
    model_n_gpu_layers = (
        spec.n_gpu_layers if spec.n_gpu_layers is not None else args.n_gpu_layers
    )
    if n_gpu_layers_override is not None:
        model_n_gpu_layers = n_gpu_layers_override

    cmd = [
        sys.executable,
        benchmark_script,
        "--model",
        spec.path,
        "--model-name",
        spec.name,
        "--num-prompts",
        str(args.num_prompts),
        "--max-tokens",
        str(model_max_tokens),
        "--n-ctx",
        str(model_n_ctx),
        "--n-gpu-layers",
        str(model_n_gpu_layers),
        "--n-threads",
        str(args.n_threads),
        "--temperature",
        str(args.temperature),
        "--top-p",
        str(args.top_p),
        "--seed",
        str(args.seed),
        "--json-out",
        json_out,
    ]

    if args.prompts_file:
        cmd.extend(["--prompts-file", args.prompts_file])
    if args.warmup:
        cmd.append("--warmup")

    return cmd


def build_gpu_retry_schedule(initial_layers: int) -> List[int]:
    """Create a conservative retry schedule that degrades GPU offload on failure."""
    schedule = [max(0, initial_layers)]

    current = max(0, initial_layers)
    while current > 0:
        # Large step down first, then converge to CPU-only fallback.
        current = max(0, current - 4)
        if current not in schedule:
            schedule.append(current)

    return schedule


def format_optional_float(value: Optional[float], precision: int = 3) -> str:
    if value is None:
        return ""
    return f"{value:.{precision}f}"


def main() -> int:
    args = parse_args()
    specs = load_model_specs(args.models_file)
    benchmark_script = os.path.join(os.path.dirname(__file__), "benchmark.py")

    results = []

    for idx, spec in enumerate(specs, start=1):
        print(f"\n=== [{idx}/{len(specs)}] Benchmarking: {spec.name} ===")
        started = time.perf_counter()

        with tempfile.NamedTemporaryFile(
            prefix="bench_", suffix=".json", delete=False
        ) as temp_file:
            json_out = temp_file.name

        initial_n_gpu_layers = (
            spec.n_gpu_layers if spec.n_gpu_layers is not None else args.n_gpu_layers
        )
        retry_schedule = build_gpu_retry_schedule(initial_n_gpu_layers)

        proc = None
        selected_n_gpu_layers = None

        for attempt, n_gpu_layers in enumerate(retry_schedule, start=1):
            if attempt > 1:
                print(
                    f"Retrying {spec.name} with lower n_gpu_layers={n_gpu_layers} "
                    f"(attempt {attempt}/{len(retry_schedule)})"
                )

            cmd = build_cmd(
                benchmark_script=benchmark_script,
                spec=spec,
                args=args,
                json_out=json_out,
                n_gpu_layers_override=n_gpu_layers,
            )
            proc = subprocess.run(cmd, text=True, capture_output=True)

            if proc.stdout:
                print(proc.stdout.strip())
            if proc.stderr:
                print(proc.stderr.strip(), file=sys.stderr)

            if proc.returncode == 0:
                selected_n_gpu_layers = n_gpu_layers
                break

        elapsed = time.perf_counter() - started

        if proc is None or proc.returncode != 0:
            error_row = {
                "name": spec.name,
                "path": spec.path,
                "status": "failed",
                "error": f"benchmark.py exited with code {proc.returncode}",
                "model_load_time_s": "",
                "avg_ttft_s": "",
                "avg_tps": "",
                "peak_process_rss_mb": "",
                "avg_gpu_util_pct": "",
                "avg_power_w": "",
                "peak_ram_vram_used_mb": "",
                "peak_ram_vram_used_pct": "",
                "peak_gpu_temp_c": "",
                "peak_cpu_temp_c": "",
                "elapsed_wall_s": f"{elapsed:.3f}",
            }
            results.append(error_row)
            print(f"FAILED: {spec.name}")
            if args.fail_fast:
                break
            continue

        try:
            with open(json_out, "r", encoding="utf-8") as handle:
                summary = json.load(handle)
        except Exception as exc:
            error_row = {
                "name": spec.name,
                "path": spec.path,
                "status": "failed",
                "error": f"failed to read JSON summary: {exc}",
                "model_load_time_s": "",
                "avg_ttft_s": "",
                "avg_tps": "",
                "peak_process_rss_mb": "",
                "avg_gpu_util_pct": "",
                "avg_power_w": "",
                "peak_ram_vram_used_mb": "",
                "peak_ram_vram_used_pct": "",
                "peak_gpu_temp_c": "",
                "peak_cpu_temp_c": "",
                "elapsed_wall_s": f"{elapsed:.3f}",
            }
            results.append(error_row)
            print(f"FAILED: {spec.name}")
            if args.fail_fast:
                break
            continue
        finally:
            try:
                os.remove(json_out)
            except OSError:
                pass

        ok_row = {
            "name": summary.get("model_name", spec.name),
            "path": summary.get("model_path", spec.path),
            "status": "ok",
            "error": "",
            "model_load_time_s": f"{summary.get('model_load_time_s', 0.0):.3f}",
            "avg_ttft_s": f"{summary.get('avg_ttft_s', 0.0):.3f}",
            "avg_tps": f"{summary.get('avg_tps', 0.0):.3f}",
            "peak_process_rss_mb": f"{summary.get('memory', {}).get('peak_process_rss_mb', 0.0):.2f}",
            "avg_gpu_util_pct": format_optional_float(
                summary.get("jetson_hardware", {}).get("avg_gpu_util_pct"),
                precision=3,
            ),
            "avg_power_w": format_optional_float(
                summary.get("jetson_hardware", {}).get("avg_power_w"),
                precision=3,
            ),
            "peak_ram_vram_used_mb": format_optional_float(
                summary.get("jetson_hardware", {}).get("peak_ram_vram_used_mb"),
                precision=2,
            ),
            "peak_ram_vram_used_pct": format_optional_float(
                summary.get("jetson_hardware", {}).get("peak_ram_vram_used_pct"),
                precision=2,
            ),
            "peak_gpu_temp_c": format_optional_float(
                summary.get("jetson_hardware", {}).get("peak_gpu_temp_c"),
                precision=2,
            ),
            "peak_cpu_temp_c": format_optional_float(
                summary.get("jetson_hardware", {}).get("peak_cpu_temp_c"),
                precision=2,
            ),
            "elapsed_wall_s": f"{elapsed:.3f}",
        }
        results.append(ok_row)
        if selected_n_gpu_layers is not None and selected_n_gpu_layers != initial_n_gpu_layers:
            print(
                f"Recovered {spec.name} by lowering n_gpu_layers "
                f"{initial_n_gpu_layers} -> {selected_n_gpu_layers}"
            )
        print(
            f"OK: {ok_row['name']} | TTFT={ok_row['avg_ttft_s']}s | "
            f"TPS={ok_row['avg_tps']} | PeakRSS={ok_row['peak_process_rss_mb']}MB | "
            f"GPU%={ok_row['avg_gpu_util_pct'] or 'N/A'} | "
            f"PowerW={ok_row['avg_power_w'] or 'N/A'}"
        )

    out_fields = [
        "name",
        "path",
        "status",
        "error",
        "model_load_time_s",
        "avg_ttft_s",
        "avg_tps",
        "peak_process_rss_mb",
        "avg_gpu_util_pct",
        "avg_power_w",
        "peak_ram_vram_used_mb",
        "peak_ram_vram_used_pct",
        "peak_gpu_temp_c",
        "peak_cpu_temp_c",
        "elapsed_wall_s",
    ]

    with open(args.output_csv, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=out_fields)
        writer.writeheader()
        writer.writerows(results)

    ok_results = [r for r in results if r["status"] == "ok"]
    ok_sorted = sorted(ok_results, key=lambda row: float(row["avg_tps"]), reverse=True)

    print("\n=== Comparison Summary (sorted by TPS desc) ===")
    if not ok_sorted:
        print("No successful benchmark runs.")
    else:
        for i, row in enumerate(ok_sorted, start=1):
            print(
                f"{i}. {row['name']}: TPS={row['avg_tps']} | TTFT={row['avg_ttft_s']}s | PeakRSS={row['peak_process_rss_mb']}MB"
            )

    print(f"\nSaved comparison CSV: {args.output_csv}")
    return 0


if __name__ == "__main__":
    try:
        raise SystemExit(main())
    except KeyboardInterrupt:
        print("\nInterrupted by user.", file=sys.stderr)
        raise SystemExit(130)


## File: plot_results.py

In [ ]:
#!/usr/bin/env python3
"""Create benchmark comparison plots from outputs/comparison_results.csv."""

from __future__ import annotations

import argparse
import csv
import math
import os
from typing import Dict, List, Optional


METRIC_SPECS = [
    {
        "key": "avg_tps",
        "title": "Average TPS (higher is better)",
        "ylabel": "tokens/s",
        "higher_is_better": True,
        "always_show": True,
    },
    {
        "key": "avg_ttft_s",
        "title": "Average TTFT (lower is better)",
        "ylabel": "seconds",
        "higher_is_better": False,
        "always_show": True,
    },
    {
        "key": "peak_process_rss_mb",
        "title": "Peak Process RSS",
        "ylabel": "MB",
        "higher_is_better": False,
        "always_show": True,
    },
    {
        "key": "model_load_time_s",
        "title": "Model Load Time",
        "ylabel": "seconds",
        "higher_is_better": False,
        "always_show": True,
    },
    {
        "key": "avg_gpu_util_pct",
        "title": "Average GPU Utilization",
        "ylabel": "%",
        "higher_is_better": True,
        "always_show": False,
    },
    {
        "key": "avg_power_w",
        "title": "Average Power Draw",
        "ylabel": "W",
        "higher_is_better": False,
        "always_show": False,
    },
    {
        "key": "peak_ram_vram_used_mb",
        "title": "Peak Unified RAM/VRAM",
        "ylabel": "MB",
        "higher_is_better": False,
        "always_show": False,
    },
    {
        "key": "peak_gpu_temp_c",
        "title": "Peak GPU Temperature",
        "ylabel": "C",
        "higher_is_better": False,
        "always_show": False,
    },
    {
        "key": "peak_cpu_temp_c",
        "title": "Peak CPU Temperature",
        "ylabel": "C",
        "higher_is_better": False,
        "always_show": False,
    },
]

METRIC_BY_KEY = {spec["key"]: spec for spec in METRIC_SPECS}


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Plot benchmark comparison results.")
    parser.add_argument(
        "--input-csv",
        default="outputs/comparison_results.csv",
        help="Input CSV generated by compare_models.py",
    )
    parser.add_argument(
        "--output-png",
        default="outputs/comparison_results.png",
        help="Output image path",
    )
    parser.add_argument("--dpi", type=int, default=170, help="PNG DPI")
    parser.add_argument(
        "--sort-by",
        default="avg_tps",
        choices=[spec["key"] for spec in METRIC_SPECS],
        help="Metric used for sorting bars",
    )
    return parser.parse_args()


def _to_float(value: str, fallback: float = 0.0) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return fallback


def _to_optional_float(value: Optional[str]) -> Optional[float]:
    if value is None:
        return None
    if isinstance(value, str) and not value.strip():
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def load_rows(csv_path: str) -> List[Dict[str, str]]:
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    with open(csv_path, "r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)

    if not rows:
        raise ValueError("CSV is empty.")

    ok_rows = [row for row in rows if row.get("status") == "ok"]
    if not ok_rows:
        raise ValueError("No successful rows found (status=ok).")

    return ok_rows


def sort_rows(rows: List[Dict[str, str]], sort_by: str) -> List[Dict[str, str]]:
    reverse = METRIC_BY_KEY[sort_by]["higher_is_better"]
    return sorted(rows, key=lambda r: _to_float(r.get(sort_by, "0")), reverse=reverse)


def select_metrics(rows: List[Dict[str, str]]) -> List[Dict[str, object]]:
    selected: List[Dict[str, object]] = []
    for spec in METRIC_SPECS:
        if spec["always_show"]:
            selected.append(spec)
            continue

        key = str(spec["key"])
        has_any = any(_to_optional_float(row.get(key)) is not None for row in rows)
        if has_any:
            selected.append(spec)

    return selected


def render_plot(rows: List[Dict[str, str]], output_png: str, dpi: int) -> None:
    try:
        import matplotlib.pyplot as plt
    except ImportError as exc:
        raise RuntimeError(
            "matplotlib is not installed. Run: pip install matplotlib"
        ) from exc

    names = [r.get("name", "unknown") for r in rows]
    metrics = select_metrics(rows)

    ncols = 3
    nrows = max(1, math.ceil(len(metrics) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.8 * nrows))
    fig.suptitle("Local LLM Comparison (Jetson Orin Nano)", fontsize=15, fontweight="bold")

    bar_color = "#2f6f8f"

    axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for idx, spec in enumerate(metrics):
        ax = axes_flat[idx]
        key = str(spec["key"])
        values = [_to_float(r.get(key, "0")) for r in rows]
        ax.bar(names, values, color=bar_color)
        ax.set_title(str(spec["title"]))
        ax.set_ylabel(str(spec["ylabel"]))
        ax.tick_params(axis="x", labelrotation=20)
        ax.grid(axis="y", alpha=0.25)

    for idx in range(len(metrics), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(output_png, dpi=dpi)
    plt.close(fig)


def main() -> int:
    args = parse_args()
    rows = load_rows(args.input_csv)
    rows = sort_rows(rows, args.sort_by)
    render_plot(rows, args.output_png, args.dpi)

    print(f"Saved plot: {args.output_png}")
    print("Tip: compare throughput, latency, memory, and Jetson hardware metrics together.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## File: rank_models.py

In [ ]:
#!/usr/bin/env python3
"""Rank benchmarked models using weighted normalized metrics.

Default score weights:
- TPS (higher is better): 0.50
- Peak RSS (lower is better): 0.30
- TTFT (lower is better): 0.20
"""

from __future__ import annotations

import argparse
import csv
import os
from typing import Dict, List, Optional, Tuple


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Rank models from outputs/comparison_results.csv")
    parser.add_argument(
        "--input-csv",
        default="outputs/comparison_results.csv",
        help="Input CSV generated by compare_models.py",
    )
    parser.add_argument(
        "--output-csv",
        default="outputs/comparison_ranked.csv",
        help="Output ranked CSV path",
    )
    parser.add_argument(
        "--w-tps",
        type=float,
        default=0.50,
        help="Weight for avg_tps (higher is better)",
    )
    parser.add_argument(
        "--w-rss",
        type=float,
        default=0.30,
        help="Weight for peak_process_rss_mb (lower is better)",
    )
    parser.add_argument(
        "--w-ttft",
        type=float,
        default=0.20,
        help="Weight for avg_ttft_s (lower is better)",
    )
    parser.add_argument(
        "--w-gpu-util",
        type=float,
        default=0.0,
        help="Weight for avg_gpu_util_pct (higher is better)",
    )
    parser.add_argument(
        "--w-power",
        type=float,
        default=0.0,
        help="Weight for avg_power_w (lower is better)",
    )
    parser.add_argument(
        "--w-jetson-ram",
        type=float,
        default=0.0,
        help="Weight for peak_ram_vram_used_mb (lower is better)",
    )
    parser.add_argument(
        "--w-gpu-temp",
        type=float,
        default=0.0,
        help="Weight for peak_gpu_temp_c (lower is better)",
    )
    parser.add_argument(
        "--w-cpu-temp",
        type=float,
        default=0.0,
        help="Weight for peak_cpu_temp_c (lower is better)",
    )
    return parser.parse_args()


def to_float(value: str, default: float = 0.0) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def to_optional_float(value: Optional[str]) -> Optional[float]:
    if value is None:
        return None
    if isinstance(value, str) and not value.strip():
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def min_max_norm(value: float, min_v: float, max_v: float, higher_is_better: bool) -> float:
    if max_v - min_v == 0:
        return 1.0
    if higher_is_better:
        return (value - min_v) / (max_v - min_v)
    return (max_v - value) / (max_v - min_v)


def load_ok_rows(path: str) -> List[Dict[str, str]]:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"CSV not found: {path}")

    with open(path, "r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))

    ok_rows = [row for row in rows if row.get("status") == "ok"]
    if not ok_rows:
        raise ValueError("No successful benchmark rows found (status=ok).")

    return ok_rows


def validate_weights(weight_map: Dict[str, float]) -> None:
    for name, value in weight_map.items():
        if value < 0:
            raise ValueError(f"{name} must be >= 0")
    total = sum(weight_map.values())
    if total <= 0:
        raise ValueError("At least one weight must be > 0")


def _compute_norms(
    rows: List[Dict[str, str]],
    key: str,
    higher_is_better: bool,
) -> Tuple[List[float], bool]:
    values = [to_optional_float(row.get(key)) for row in rows]
    present = [v for v in values if v is not None]
    if not present:
        return [0.0 for _ in rows], False

    min_v, max_v = min(present), max(present)
    norms: List[float] = []
    for value in values:
        if value is None:
            norms.append(0.0)
        else:
            norms.append(min_max_norm(value, min_v, max_v, higher_is_better))

    return norms, True


def rank_rows(
    rows: List[Dict[str, str]],
    weight_map: Dict[str, float],
) -> Tuple[List[Dict[str, str]], List[str]]:
    metric_specs = [
        ("w_tps", "avg_tps", "norm_tps", True),
        ("w_rss", "peak_process_rss_mb", "norm_rss", False),
        ("w_ttft", "avg_ttft_s", "norm_ttft", False),
        ("w_gpu_util", "avg_gpu_util_pct", "norm_gpu_util", True),
        ("w_power", "avg_power_w", "norm_power", False),
        ("w_jetson_ram", "peak_ram_vram_used_mb", "norm_jetson_ram", False),
        ("w_gpu_temp", "peak_gpu_temp_c", "norm_gpu_temp", False),
        ("w_cpu_temp", "peak_cpu_temp_c", "norm_cpu_temp", False),
    ]

    scored = [dict(row) for row in rows]
    for row in scored:
        row["score"] = "0.000000"
        for _, _, norm_key, _ in metric_specs:
            row[norm_key] = ""

    accum_scores = [0.0 for _ in rows]
    active_weight_sum = 0.0
    used_metrics: List[str] = []

    for weight_name, metric_key, norm_key, higher_is_better in metric_specs:
        weight = weight_map.get(weight_name, 0.0)
        if weight <= 0:
            continue

        norms, has_data = _compute_norms(rows, metric_key, higher_is_better)
        if not has_data:
            continue

        used_metrics.append(weight_name)
        active_weight_sum += weight
        for i, norm_value in enumerate(norms):
            scored[i][norm_key] = f"{norm_value:.6f}"
            accum_scores[i] += weight * norm_value

    if active_weight_sum <= 0:
        raise ValueError(
            "No metrics with usable data were selected. "
            "Check CSV hardware columns or reduce hardware weights."
        )

    for i, row in enumerate(scored):
        row["score"] = f"{(accum_scores[i] / active_weight_sum):.6f}"

    scored.sort(key=lambda r: to_float(r["score"]), reverse=True)

    for idx, row in enumerate(scored, start=1):
        row["rank"] = str(idx)

    return scored, used_metrics


def save_rows(path: str, rows: List[Dict[str, str]]) -> None:
    fields = [
        "rank",
        "name",
        "path",
        "status",
        "error",
        "score",
        "norm_tps",
        "norm_rss",
        "norm_ttft",
        "norm_gpu_util",
        "norm_power",
        "norm_jetson_ram",
        "norm_gpu_temp",
        "norm_cpu_temp",
        "model_load_time_s",
        "avg_ttft_s",
        "avg_tps",
        "peak_process_rss_mb",
        "avg_gpu_util_pct",
        "avg_power_w",
        "peak_ram_vram_used_mb",
        "peak_ram_vram_used_pct",
        "peak_gpu_temp_c",
        "peak_cpu_temp_c",
        "elapsed_wall_s",
    ]
    with open(path, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def main() -> int:
    args = parse_args()
    weight_map = {
        "w_tps": args.w_tps,
        "w_rss": args.w_rss,
        "w_ttft": args.w_ttft,
        "w_gpu_util": args.w_gpu_util,
        "w_power": args.w_power,
        "w_jetson_ram": args.w_jetson_ram,
        "w_gpu_temp": args.w_gpu_temp,
        "w_cpu_temp": args.w_cpu_temp,
    }
    validate_weights(weight_map)
    rows = load_ok_rows(args.input_csv)
    ranked, used_metrics = rank_rows(rows, weight_map)
    save_rows(args.output_csv, ranked)

    print("=== Weighted Ranking ===")
    print(
        "Weights -> "
        f"TPS: {args.w_tps:.2f}, RSS: {args.w_rss:.2f}, TTFT: {args.w_ttft:.2f}, "
        f"GPU Util: {args.w_gpu_util:.2f}, Power: {args.w_power:.2f}, "
        f"Jetson RAM: {args.w_jetson_ram:.2f}, GPU Temp: {args.w_gpu_temp:.2f}, "
        f"CPU Temp: {args.w_cpu_temp:.2f}"
    )
    print(f"Active metrics: {', '.join(used_metrics)}")
    for row in ranked:
        print(
            f"{row['rank']}. {row['name']} | score={row['score']} | TPS={row['avg_tps']} | "
            f"TTFT={row['avg_ttft_s']}s | PeakRSS={row['peak_process_rss_mb']}MB | "
            f"Power={row.get('avg_power_w') or 'N/A'}W | "
            f"GPUtemp={row.get('peak_gpu_temp_c') or 'N/A'}C"
        )

    print(f"Saved ranked CSV: {args.output_csv}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
